In [ ]:
# 1. Environment & Path Setup
import os
import sys

# Add the project root directory to sys.path so 'pipeline' is importable
sys.path.append(os.path.abspath('..'))

import numpy as np  # noqa: F401
import pandas as pd  # noqa: F401

import duckdb
DB_PATH = '/home/kishor/Projects/ShotDist/shotdistributionpredictor/data/events.db'

con = duckdb.connect(DB_PATH, read_only=True)

In [ ]:
pm_df = con.execute("SELECT * FROM opspatial WHERE total_reception_events >= 15 AND total_release_events >= 15").df()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

import matplotlib.pyplot as plt

feature_cols = [
    "average_reception_x",
    "average_reception_norm_y",
    "average_release_x",
    "average_release_norm_y",
]

pm_X = pm_df[feature_cols].dropna()
pm_X_scaled = StandardScaler().fit_transform(pm_X)

pm_k_range = range(2, 11)
pm_inertia = []
pm_silhouette_scores = []

for n_clusters in pm_k_range:
    model = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
    labels = model.fit_predict(pm_X_scaled)

    pm_inertia.append(model.inertia_)
    pm_silhouette_scores.append(
        silhouette_score(
            pm_X_scaled,
            labels,
            sample_size=min(5000, len(pm_X_scaled)),
            random_state=42,
        )
    )

fig_pm, (ax1_pm, ax2_pm) = plt.subplots(1, 2, figsize=(14, 5))

ax1_pm.plot(list(pm_k_range), pm_inertia, marker="o")
ax1_pm.set(title="Elbow Method", xlabel="Number of Clusters (k)", ylabel="Inertia")

ax2_pm.plot(list(pm_k_range), pm_silhouette_scores, marker="o")
ax2_pm.set(title="Silhouette Scores", xlabel="Number of Clusters (k)", ylabel="Silhouette Score")

plt.tight_layout()
plt.show()

In [ ]:
pm_kmeans_models = {
    k: KMeans(n_clusters=k, n_init=10, random_state=42).fit(pm_X_scaled)
    for k in (3, 7)
}

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for row, k_value in enumerate((3, 7)):
    labels = pm_kmeans_models[k_value].labels_

    for col, (x_col, y_col, title) in enumerate((
        ("average_reception_x", "average_reception_norm_y", "Reception"),
        ("average_release_x", "average_release_norm_y", "Release"),
    )):
        ax = axes[row, col]
        scatter = ax.scatter(
            pm_X[x_col], pm_X[y_col],
            c=labels, cmap="tab10", s=8, alpha=0.5, rasterized=True,
        )
        ax.set(
            title=f"{title} positions — k={k_value}",
            xlabel=x_col,
            ylabel=y_col,
        )
        fig.colorbar(scatter, ax=ax, ticks=range(k_value), label="Cluster")

plt.tight_layout()
plt.show()

In [ ]:
team1_idx = pm_df.loc[pm_X.index].index[
    pm_df.loc[pm_X.index, "team_id"].eq(1)
]

team1_X = pm_X.loc[team1_idx]
team1_X_scaled = pm_X_scaled[pm_X.index.get_indexer(team1_idx)]

fig_team1, axes_team1 = plt.subplots(2, 2, figsize=(12, 10))

for row, k_value in enumerate((3, 7)):
    labels_team1 = pm_kmeans_models[k_value].predict(team1_X_scaled)

    for col, (x_col, y_col, title) in enumerate((
        ("average_reception_x", "average_reception_norm_y", "Reception"),
        ("average_release_x", "average_release_norm_y", "Release"),
    )):
        ax_team1 = axes_team1[row, col]
        scatter_team1 = ax_team1.scatter(
            team1_X[x_col],
            team1_X[y_col],
            c=labels_team1,
            cmap="tab10",
            s=8,
            alpha=0.5,
            rasterized=True,
        )

        ax_team1.set(
            title=f"{title} positions — team_id=1, k={k_value}",
            xlabel=x_col,
            ylabel=y_col,
        )
        fig_team1.colorbar(
            scatter_team1,
            ax=ax_team1,
            ticks=range(k_value),
            label="Cluster",
        )

plt.tight_layout()
plt.show()

In [ ]:
cluster_model = pm_kmeans_models[7]
cluster_by_index = pd.Series(
    cluster_model.predict(pm_X_scaled),
    index=pm_X.index,
)

team1_data = pm_df.loc[pm_X.index]
team1_data = team1_data.loc[team1_data["team_id"].eq(1)].copy()
team1_data["cluster"] = cluster_by_index.loc[team1_data.index]

for match_id, game in team1_data.groupby("match_id"):
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    for ax, x_col, y_col, title in (
        (axes[0], "average_reception_x", "average_reception_norm_y", "Reception"),
        (axes[1], "average_release_x", "average_release_norm_y", "Release"),
    ):
        scatter = ax.scatter(
            game[x_col],
            game[y_col],
            c=game["cluster"],
            cmap="tab10",
            vmin=0,
            vmax=6,
            s=45,
            alpha=0.8,
        )

        for _, point in game.iterrows():
            ax.annotate(
                str(point["player"]),
                (point[x_col], point[y_col]),
                xytext=(4, 4),
                textcoords="offset points",
                fontsize=7,
            )

        ax.set(title=title, xlabel=x_col, ylabel=y_col)
        ax.grid(alpha=0.2)

    fig.suptitle(f"Team 1 — Match {match_id} (k=7 clusters)")
    fig.colorbar(scatter, ax=axes, ticks=range(7), label="Cluster")
    plt.tight_layout()
    plt.show()

In [ ]:
from matplotlib.colors import Normalize

cmap = plt.get_cmap("tab10")
norm = Normalize(vmin=-0.5, vmax=6.5)

for match_id, game in team1_data.groupby("match_id"):
    fig, ax = plt.subplots(figsize=(10, 8))

    for _, point in game.iterrows():
        cluster = int(point["cluster"])
        color = cmap(norm(cluster))

        reception = (
            point["average_reception_x"],
            point["average_reception_norm_y"],
        )
        release = (
            point["average_release_x"],
            point["average_release_norm_y"],
        )

        ax.plot(
            [reception[0], release[0]],
            [reception[1], release[1]],
            color=color,
            alpha=0.6,
            linewidth=1,
        )
        ax.scatter(*reception, color=color, marker="o", s=45)
        ax.scatter(*release, color=color, marker="x", s=55)

        midpoint = (
            (reception[0] + release[0]) / 2,
            (reception[1] + release[1]) / 2,
        )
        ax.annotate(
            str(point["player"]),
            midpoint,
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=7,
        )

    ax.set(
        title=f"Team 1 — Match {match_id} (k=7 clusters)",
        xlabel="Average x",
        ylabel="Average normalized y",
    )
    ax.grid(alpha=0.2)

    mappable = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    mappable.set_array([])
    fig.colorbar(mappable, ax=ax, ticks=range(7), label="Cluster")

    ax.scatter([], [], color="black", marker="o", label="Reception")
    ax.scatter([], [], color="black", marker="x", label="Release")
    ax.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
pm_y_feature_cols = [
    "average_reception_x",
    "average_reception_y",
    "average_release_x",
    "average_release_y",
]

pm_y_X = pm_df[pm_y_feature_cols].dropna()
pm_y_X_scaled = StandardScaler().fit_transform(pm_y_X)

pm_y_k_range = range(2, 21)
pm_y_inertia = []
pm_y_silhouette_scores = []

for pm_y_k in pm_y_k_range:
    pm_y_model = KMeans(n_clusters=pm_y_k, n_init=10, random_state=42)
    pm_y_labels = pm_y_model.fit_predict(pm_y_X_scaled)

    pm_y_inertia.append(pm_y_model.inertia_)
    pm_y_silhouette_scores.append(
        silhouette_score(
            pm_y_X_scaled,
            pm_y_labels,
            sample_size=min(5000, len(pm_y_X_scaled)),
            random_state=42,
        )
    )

fig_pm_y, (ax1_pm_y, ax2_pm_y) = plt.subplots(1, 2, figsize=(14, 5))

ax1_pm_y.plot(list(pm_y_k_range), pm_y_inertia, marker="o")
ax1_pm_y.set(title="Elbow Method", xlabel="Number of Clusters (k)", ylabel="Inertia")

ax2_pm_y.plot(list(pm_y_k_range), pm_y_silhouette_scores, marker="o")
ax2_pm_y.set(
    title="Silhouette Scores",
    xlabel="Number of Clusters (k)",
    ylabel="Silhouette Score",
)

plt.tight_layout()
plt.show()

In [ ]:
pm_y_feature_cols = [
    "average_reception_x",
    "average_reception_y",
    "average_release_x",
    "average_release_y",
]

pm_y_df = pm_df[pm_y_feature_cols].copy()
pm_y_df["average_reception_y"] = pm_y_df["average_reception_y"].abs()
pm_y_df["average_release_y"] = pm_y_df["average_release_y"].abs()

pm_y_X = pm_y_df.dropna()
pm_y_X_scaled = StandardScaler().fit_transform(pm_y_X)

pm_y_k_range = range(2, 11)
pm_y_inertia = []
pm_y_silhouette_scores = []

for pm_y_k in pm_y_k_range:
    pm_y_model = KMeans(n_clusters=pm_y_k, n_init=10, random_state=42)
    pm_y_labels = pm_y_model.fit_predict(pm_y_X_scaled)

    pm_y_inertia.append(pm_y_model.inertia_)
    pm_y_silhouette_scores.append(
        silhouette_score(
            pm_y_X_scaled,
            pm_y_labels,
            sample_size=min(5000, len(pm_y_X_scaled)),
            random_state=42,
        )
    )

fig_pm_y, (ax1_pm_y, ax2_pm_y) = plt.subplots(1, 2, figsize=(14, 5))

ax1_pm_y.plot(list(pm_y_k_range), pm_y_inertia, marker="o")
ax1_pm_y.set(
    title="Elbow Method (abs avg y)",
    xlabel="Number of Clusters (k)",
    ylabel="Inertia",
)

ax2_pm_y.plot(list(pm_y_k_range), pm_y_silhouette_scores, marker="o")
ax2_pm_y.set(
    title="Silhouette Scores (abs avg y)",
    xlabel="Number of Clusters (k)",
    ylabel="Silhouette Score",
)

plt.tight_layout()
plt.show()

In [ ]:
pm_y_feature_cols = [
    "average_reception_x",
    "average_reception_y",
    "average_release_x",
    "average_release_y",
]

pm_y_df = pm_df[pm_y_feature_cols].copy()
pm_y_df["average_reception_y"] = pm_y_df["average_reception_y"].abs()
pm_y_df["average_release_y"] = pm_y_df["average_release_y"].abs()

pm_y_X = pm_y_df.dropna()
pm_y_X_scaled = StandardScaler().fit_transform(pm_y_X)

pm_y_k_range = range(2, 11)
pm_y_inertia = []
pm_y_silhouette_scores = []

for pm_y_k in pm_y_k_range:
    pm_y_model = KMeans(n_clusters=pm_y_k, n_init=10, random_state=42)
    pm_y_labels = pm_y_model.fit_predict(pm_y_X_scaled)

    pm_y_inertia.append(pm_y_model.inertia_)
    pm_y_silhouette_scores.append(
        silhouette_score(
            pm_y_X_scaled,
            pm_y_labels,
            sample_size=min(5000, len(pm_y_X_scaled)),
            random_state=42,
        )
    )

fig_pm_y, (ax1_pm_y, ax2_pm_y) = plt.subplots(1, 2, figsize=(14, 5))

ax1_pm_y.plot(list(pm_y_k_range), pm_y_inertia, marker="o")
ax1_pm_y.set(
    title="Elbow Method (abs avg y)",
    xlabel="Number of Clusters (k)",
    ylabel="Inertia",
)

ax2_pm_y.plot(list(pm_y_k_range), pm_y_silhouette_scores, marker="o")
ax2_pm_y.set(
    title="Silhouette Scores (abs avg y)",
    xlabel="Number of Clusters (k)",
    ylabel="Silhouette Score",
)

plt.tight_layout()
plt.show()

In [ ]:

pm_df = pm_df.copy()

pm_df["avg_reception_simpnorm_y"] = np.where(
    pm_df["average_reception_y"] <= 40,
    pm_df["average_reception_y"],
    80 - pm_df["average_reception_y"],
)

pm_df["avg_release_simpnorm_y"] = np.where(
    pm_df["average_release_y"] <= 40,
    pm_df["average_release_y"],
    80 - pm_df["average_release_y"],
)

pm_df["avg_reception_simpnorm_y_abs"] = pm_df["avg_reception_simpnorm_y"].abs()
pm_df["avg_release_simpnorm_y_abs"] = pm_df["avg_release_simpnorm_y"].abs()

feature_cols = [
    "average_reception_x",
    "avg_reception_simpnorm_y_abs",
    "average_release_x",
    "avg_release_simpnorm_y_abs",
]

cluster_df = pm_df[feature_cols].dropna().copy()
cluster_X = StandardScaler().fit_transform(cluster_df)

k_value = 7
cluster_model = KMeans(n_clusters=k_value, n_init=10, random_state=42)
cluster_labels = cluster_model.fit_predict(cluster_X)

cluster_df["cluster"] = cluster_labels

cluster_counts = cluster_df["cluster"].value_counts().sort_index()
print("Cluster counts:")
print(cluster_counts)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

scatter1 = axes[0].scatter(
    cluster_df["average_reception_x"],
    cluster_df["avg_reception_simpnorm_y_abs"],
    c=cluster_df["cluster"],
    cmap="tab10",
    s=25,
    alpha=0.7,
)
axes[0].set(
    title=f"Reception clusters — k={k_value}",
    xlabel="average_reception_x",
    ylabel="avg_reception_simpnorm_y_abs",
)
fig.colorbar(scatter1, ax=axes[0], ticks=range(k_value), label="Cluster")

scatter2 = axes[1].scatter(
    cluster_df["average_release_x"],
    cluster_df["avg_release_simpnorm_y_abs"],
    c=cluster_df["cluster"],
    cmap="tab10",
    s=25,
    alpha=0.7,
)
axes[1].set(
    title=f"Release clusters — k={k_value}",
    xlabel="average_release_x",
    ylabel="avg_release_simpnorm_y_abs",
)
fig.colorbar(scatter2, ax=axes[1], ticks=range(k_value), label="Cluster")

plt.tight_layout()
plt.show()


In [ ]:
k_value = 7

feature_cols = [
    "average_reception_x",
    "avg_reception_simpnorm_y_abs",
    "average_release_x",
    "avg_release_simpnorm_y_abs",
]

team1_df = pm_df.loc[pm_df["team_id"].eq(1)].copy()
team1_features = team1_df[feature_cols].dropna().copy()

scaler = StandardScaler()
team1_scaled = scaler.fit_transform(team1_features)

team1_model = KMeans(n_clusters=k_value, n_init=10, random_state=42)
team1_features["cluster"] = team1_model.fit_predict(team1_scaled)

team1_df = team1_df.loc[team1_features.index].copy()
team1_df["cluster"] = team1_features["cluster"]

plt.close("all")

for match_id, game in team1_df.groupby("match_id"):
    fig, ax = plt.subplots(figsize=(10, 8))

    for _, point in game.iterrows():
        cluster = int(point["cluster"])
        color = plt.get_cmap("tab10")(cluster % 10)

        reception = (
            point["average_reception_x"],
            point["avg_reception_simpnorm_y_abs"],
        )
        release = (
            point["average_release_x"],
            point["avg_release_simpnorm_y_abs"],
        )

        ax.plot(
            [reception[0], release[0]],
            [reception[1], release[1]],
            color=color,
            alpha=0.7,
            linewidth=1.5,
        )
        ax.scatter(*reception, color=color, marker="o", s=45)
        ax.scatter(*release, color=color, marker="x", s=55)

        midpoint = (
            (reception[0] + release[0]) / 2,
            (reception[1] + release[1]) / 2,
        )
        ax.annotate(
            str(point["player"]),
            midpoint,
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=7,
        )

    ax.set(
        title=f"Team 1 — Match {match_id} (k={k_value} clusters)",
        xlabel="Average x",
        ylabel="Abs symmetric-normalized y",
    )
    ax.grid(alpha=0.2)

    legend_handles = [
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            label="Reception",
            markerfacecolor="black",
            markersize=8,
        ),
        plt.Line2D(
            [0],
            [0],
            marker="x",
            color="w",
            label="Release",
            markersize=8,
        ),
    ]
    ax.legend(handles=legend_handles)

    plt.tight_layout()
    plt.show()
    plt.close(fig)

In [ ]:
from mplsoccer import Pitch
from matplotlib.lines import Line2D

k_value = 7

feature_cols = [
    "average_reception_x",
    "avg_reception_simpnorm_y_abs",
    "average_release_x",
    "avg_release_simpnorm_y_abs",
]

team1_df = pm_df.loc[pm_df["team_id"].eq(1)].copy()
team1_features = team1_df[feature_cols].dropna().copy()

scaler = StandardScaler()
team1_scaled = scaler.fit_transform(team1_features)

team1_model = KMeans(n_clusters=k_value, n_init=10, random_state=42)
team1_features["cluster"] = team1_model.fit_predict(team1_scaled)

team1_df = team1_df.loc[team1_features.index].copy()
team1_df["cluster"] = team1_features["cluster"]

cluster_cmap = plt.get_cmap("tab10", k_value)

for match_id, game in team1_df.groupby("match_id"):
    pitch = Pitch(
        pitch_type="custom",
        pitch_length=120,
        pitch_width=40,
        pitch_color="white",
        line_color="black",
        axis=True,
        label=True,
    )
    fig, ax = pitch.draw(figsize=(14, 6))
    fig.subplots_adjust(right=0.78)

    for _, point in game.iterrows():
        cluster = int(point["cluster"])
        color = cluster_cmap(cluster)

        reception = (
            point["average_reception_x"],
            point["avg_reception_simpnorm_y_abs"],
        )
        release = (
            point["average_release_x"],
            point["avg_release_simpnorm_y_abs"],
        )

        ax.plot(
            [reception[0], release[0]],
            [reception[1], release[1]],
            color=color,
            alpha=0.7,
            linewidth=1.5,
        )
        ax.scatter(*reception, color=color, marker="o", s=45, zorder=3)
        ax.scatter(*release, color=color, marker="x", s=55, zorder=3)

        midpoint = (
            (reception[0] + release[0]) / 2,
            (reception[1] + release[1]) / 2,
        )
        ax.annotate(
            str(point["player"]),
            midpoint,
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=7,
        )

    ax.set_xlim(0, 120)
    ax.set_ylim(40, 0)
    ax.set_xticks(np.arange(0, 121, 20))
    ax.set_yticks(np.arange(0, 41, 10))
    ax.set_xlabel("Average x")
    ax.set_ylabel("Symmetric-normalized y")

    cluster_handles = [
        Line2D(
            [0],
            [0],
            color=cluster_cmap(cluster),
            linewidth=2,
            label=f"Cluster {cluster}",
        )
        for cluster in range(k_value)
    ]
    point_handles = [
        Line2D(
            [0], [0], color="black", marker="o", linestyle="None",
            label="Reception",
        ),
        Line2D(
            [0], [0], color="black", marker="x", linestyle="None",
            label="Release",
        ),
    ]

    ax.legend(
        handles=cluster_handles + point_handles,
        title="Clusters and points",
        loc="upper left",
        bbox_to_anchor=(1.02, 1),
    )

    ax.set_title(f"Team 1 — Match {match_id} (k={k_value} clusters)")
    fig.tight_layout()
    plt.show()
    plt.close(fig)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from mplsoccer import Pitch
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

k_value = 7

feature_cols = [
    "average_reception_x",
    "avg_reception_simpnorm_y_abs",
    "average_release_x",
    "avg_release_simpnorm_y_abs",
]

# Fit the scaler and KMeans model on the full dataset.
cluster_features = pm_df[feature_cols].dropna().copy()
scaler = StandardScaler()
cluster_scaled = scaler.fit_transform(cluster_features)

kmeans = KMeans(n_clusters=k_value, n_init=10, random_state=42)
clustered_df = pm_df.loc[cluster_features.index].copy()
clustered_df["cluster"] = kmeans.fit_predict(cluster_scaled)

# Filter to team 1 only after clustering the full dataset.
team1_df = clustered_df.loc[clustered_df["team_id"].eq(1)].copy()
cluster_cmap = plt.get_cmap("tab10", k_value)

for match_id, game in team1_df.groupby("match_id"):
    pitch = Pitch(
        pitch_type="custom",
        pitch_length=120,
        pitch_width=40,
        pitch_color="white",
        line_color="black",
        axis=True,
        label=True,
    )
    fig, ax = pitch.draw(figsize=(14, 6))

    for _, point in game.iterrows():
        cluster = int(point["cluster"])
        color = cluster_cmap(cluster)

        reception = (
            point["average_reception_x"],
            point["avg_reception_simpnorm_y_abs"],
        )
        release = (
            point["average_release_x"],
            point["avg_release_simpnorm_y_abs"],
        )

        ax.plot(
            [reception[0], release[0]],
            [reception[1], release[1]],
            color=color,
            alpha=0.7,
            linewidth=1.5,
        )
        ax.scatter(*reception, color=color, marker="o", s=45, zorder=3)
        ax.scatter(*release, color=color, marker="x", s=55, zorder=3)

        midpoint = (
            (reception[0] + release[0]) / 2,
            (reception[1] + release[1]) / 2,
        )
        ax.annotate(
            str(point["player"]),
            midpoint,
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=7,
        )

    ax.set_xlim(0, 120)
    ax.set_ylim(40, 0)
    ax.set_xticks(np.arange(0, 121, 20))
    ax.set_yticks(np.arange(0, 41, 10))
    ax.set_xlabel("Average x")
    ax.set_ylabel("Symmetric-normalized y")
    ax.set_title(f"Team 1 - Match {match_id} (k={k_value} clusters)")

    cluster_handles = [
        Line2D(
            [0], [0],
            color=cluster_cmap(cluster),
            linewidth=2,
            label=f"Cluster {cluster}",
        )
        for cluster in range(k_value)
    ]
    point_handles = [
        Line2D(
            [0], [0],
            color="black",
            marker="o",
            linestyle="None",
            label="Reception",
        ),
        Line2D(
            [0], [0],
            color="black",
            marker="x",
            linestyle="None",
            label="Release",
        ),
    ]

    ax.legend(
        handles=cluster_handles + point_handles,
        title="Clusters and points",
        loc="upper left",
        bbox_to_anchor=(1.02, 1),
    )

    fig.tight_layout(rect=(0, 0, 0.78, 1))
    plt.show()
    plt.close(fig)

In [30]:
con.close()